In [1]:
import pandas as pd
train_data = pd.read_csv("train.csv")
geo_data = pd.read_csv("geo_data.csv")
train_data.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'total_items',
 'total_price',
 'total_freight_value',
 'total_payment',
 'max_installments',
 'payment_count',
 'delivery_status']

In [2]:
geo_data.columns.tolist()

['order_id',
 'delivery_status',
 'customer_id',
 'customer_state',
 'seller_id',
 'seller_state',
 'customer_lat',
 'customer_lng',
 'seller_lat',
 'seller_lng']

In [3]:
train = train_data.merge(geo_data[[ 'customer_state','order_id',
 'seller_id',
 'seller_state',
 'customer_lat',
 'customer_lng',
 'seller_lat',
 'seller_lng']], on='order_id',  how='left')
train.columns.tolist()                        

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'total_items',
 'total_price',
 'total_freight_value',
 'total_payment',
 'max_installments',
 'payment_count',
 'delivery_status',
 'customer_state',
 'seller_id',
 'seller_state',
 'customer_lat',
 'customer_lng',
 'seller_lat',
 'seller_lng']

In [4]:
train.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_items,total_price,...,max_installments,payment_count,delivery_status,customer_state,seller_id,seller_state,customer_lat,customer_lng,seller_lat,seller_lng
0,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,2016-10-18 13:14:51,NaN,2016-10-20 00:00:00,2.0,72.89,...,1.0,1.0,late,RR,1554a68530182680ad5c8b042c3ab563,MG,2.813746,-60.701007,-22.430218,-46.573405
1,e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,NaN,NaN,2016-10-28 00:00:00,1.0,59.50,...,3.0,1.0,late,RS,a425f92c199eb576938df686728acd20,PR,-28.264640,-52.425421,-25.490559,-49.302027
2,809a282bbd5dbcabb6f2f724fca862ec,622e13439d6b5a0b486c435618b2679e,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,NaN,NaN,2016-09-30 00:00:00,NaN,NaN,...,2.0,1.0,late,SP,NaN,NaN,-23.198979,-45.943171,NaN,NaN
3,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00,3.0,134.97,...,NaN,NaN,late,SP,ecccfa2bb93b34a3bf033cc5d1dcdc69,PR,-20.585751,-47.863693,-25.507014,-49.275963
4,71303d7e93b399f5bcd537d124c0bcfa,b106b360fe2ef8849fbbd056f777b4d5,canceled,2016-10-02 22:07:52,2016-10-06 15:50:56,NaN,NaN,2016-10-25 00:00:00,1.0,100.00,...,1.0,1.0,late,SP,25e6ffe976bd75618accfe16cefcbd0d,SP,-23.478131,-46.711362,-23.570212,-46.710840


In [5]:
train["order_purchase_timestamp"].dtype

<StringDtype(na_value=nan)>

In [6]:
import pandas as pd
train["order_purchase_timestamp"] = pd.to_datetime(
    train["order_purchase_timestamp"]
)

In [7]:
train["month_name"] = (train["order_purchase_timestamp"].dt.month_name())

In [8]:
train[["order_purchase_timestamp", "month_name"]].head()

,order_purchase_timestamp,month_name
0,2016-09-04 21:15:19,September
1,2016-09-05 00:15:34,September
2,2016-09-13 15:24:19,September
3,2016-09-15 12:16:38,September
4,2016-10-02 22:07:52,October


In [9]:
train["day_name"] = (train["order_purchase_timestamp"].dt.day_name()) 

In [10]:
train[["order_purchase_timestamp" , "month_name" , "day_name"]].head()

,order_purchase_timestamp,month_name,day_name
0,2016-09-04 21:15:19,September,Sunday
1,2016-09-05 00:15:34,September,Monday
2,2016-09-13 15:24:19,September,Tuesday
3,2016-09-15 12:16:38,September,Thursday
4,2016-10-02 22:07:52,October,Sunday


In [11]:
import numpy as np

In [12]:
def haversine(lat1, lon1, lat2, lon2):
    
    R = 6371  # Earth radius in kilometers
    
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = (
        np.sin(dlat / 2)**2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2)**2
    )
    
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

In [13]:
train["distance_km"] = haversine(
    train["customer_lat"],
    train["customer_lng"],
    train["seller_lat"],
    train["seller_lng"]
)

In [14]:
train["distance_km"].head()

0    3198.843109
1     437.127147
2            NaN
3     565.959812
4      10.239056
Name: distance_km, dtype: float64

In [15]:
train.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  86
order_delivered_carrier_date     1456
order_delivered_customer_date    2380
order_estimated_delivery_date       0
total_items                       647
total_price                       647
total_freight_value               647
total_payment                       1
max_installments                    1
payment_count                       1
delivery_status                     0
customer_state                      0
seller_id                         647
seller_state                      647
customer_lat                      193
customer_lng                      193
seller_lat                        814
seller_lng                        814
month_name                          0
day_name                            0
distance_km                      1002
dtype: int64

In [16]:
numerical_features = ["total_items" ,"total_price" ,"total_freight_value",
                       "total_payment" ,"max_installments","payment_count",
                       "distance_km"]

In [17]:
train[numerical_features] = train[numerical_features].fillna(
    train[numerical_features].median()
)

In [18]:
train[numerical_features].isnull().sum()

total_items            0
total_price            0
total_freight_value    0
total_payment          0
max_installments       0
payment_count          0
distance_km            0
dtype: int64

In [19]:
train["seller_state"] = train["seller_state"].fillna("Unknown")

In [20]:
train["seller_state"].isnull().sum()

np.int64(0)

In [21]:
features = ["total_items" ,"total_price" ,"total_freight_value",
                       "total_payment" ,"max_installments","payment_count",
                       "distance_km","seller_state" , "customer_state","month_name","day_name"]
x = train[features]
y = train["delivery_status"]

In [22]:
x.shape , y.shape

((69608, 11), (69608,))

In [23]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [24]:
categorical_features = [
    "month_name",
    "day_name",
    "customer_state",
    "seller_state"
]

In [25]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

transformers = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numerical",
            numerical_transformer,
            numerical_features
        )
    ]
)

In [26]:
X_train_transformed = transformers.fit_transform(x)

In [27]:
X_train_transformed.shape

(69608, 77)

In [28]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ("transformer", transformers)
])

In [29]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('transformer', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['total_items','total_price','total_freight_value',...,'customer_state', 'month_name','day_name']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through.

In [30]:
features_name = transformers.get_feature_names_out()
features_name

array(['categorical__month_name_April', 'categorical__month_name_August',
       'categorical__month_name_December',
       'categorical__month_name_February',
       'categorical__month_name_January', 'categorical__month_name_July',
       'categorical__month_name_June', 'categorical__month_name_March',
       'categorical__month_name_May', 'categorical__month_name_November',
       'categorical__month_name_October',
       'categorical__month_name_September',
       'categorical__day_name_Friday', 'categorical__day_name_Monday',
       'categorical__day_name_Saturday', 'categorical__day_name_Sunday',
       'categorical__day_name_Thursday', 'categorical__day_name_Tuesday',
       'categorical__day_name_Wednesday',
       'categorical__customer_state_AC', 'categorical__customer_state_AL',
       'categorical__customer_state_AM', 'categorical__customer_state_AP',
       'categorical__customer_state_BA', 'categorical__customer_state_CE',
       'categorical__customer_state_DF', 'categor

In [31]:
import joblib 
joblib.dump(transformers , "transformer.pkl")

['transformer.pkl']

In [32]:
joblib.dump(features_name, "features_name.pkl")

['features_name.pkl']

In [33]:
X_train_transformed  = pd.DataFrame(
    X_train_transformed.toarray() if hasattr(X_train_transformed, "toarray") else X_train_transformed,
    columns= features_name
)

X_train_transformed.head()

,categorical__month_name_April,categorical__month_name_August,categorical__month_name_December,categorical__month_name_February,categorical__month_name_January,categorical__month_name_July,categorical__month_name_June,categorical__month_name_March,categorical__month_name_May,categorical__month_name_November,...,categorical__seller_state_SE,categorical__seller_state_SP,categorical__seller_state_Unknown,numerical__total_items,numerical__total_price,numerical__total_freight_value,numerical__total_payment,numerical__max_installments,numerical__payment_count,numerical__distance_km
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.595437,-0.307833,2.056068,-0.106298,-0.717307,-0.116931,4.358508
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.260091,-0.373001,-0.334003,-0.388679,0.007434,-0.116931,-0.298513
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,-0.260091,-0.246462,-0.272476,-0.546142,-0.354937,-0.116931,-0.278218
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.450965,-0.005696,-0.687662,-0.253467,-0.354937,-0.116931,-0.081265
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-0.260091,-0.175892,-0.645143,-0.230431,-0.717307,-0.116931,-1.018365


In [34]:
joblib.dump(X_train_transformed,"X_train_transformed.pkl")

['X_train_transformed.pkl']

In [35]:
train.to_csv("train_data_2.csv",index=False)

In [25]:
train.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'total_items', 'total_price', 'total_freight_value', 'total_payment',
       'max_installments', 'payment_count', 'delivery_status',
       'customer_state', 'seller_id', 'seller_state', 'customer_lat',
       'customer_lng', 'seller_lat', 'seller_lng', 'month_name', 'day_name',
       'distance_km'],
      dtype='str')

In [26]:
features = [
    "total_items",
    "total_price",
    "total_freight_value",
    "total_payment",
    "max_installments",
    "payment_count",
    "distance_km",
    "seller_state",
    "customer_state",
    "month_name",
    "day_name"
]

train[features].isna().mean().sort_values(ascending=False)

total_items            0.0
total_price            0.0
total_freight_value    0.0
total_payment          0.0
max_installments       0.0
payment_count          0.0
distance_km            0.0
seller_state           0.0
customer_state         0.0
month_name             0.0
day_name               0.0
dtype: float64

In [27]:
categorical_features = [
    "seller_state",
    "customer_state",
    "month_name",
    "day_name"
]

for col in categorical_features:
    print(f"\n{col}:")
    print(sorted(train[col].dropna().unique().tolist()))
    


seller_state:
['AC', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RS', 'SC', 'SE', 'SP', 'Unknown']

customer_state:
['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']

month_name:
['April', 'August', 'December', 'February', 'January', 'July', 'June', 'March', 'May', 'November', 'October', 'September']

day_name:
['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


In [28]:
numeric_features = [
    "total_items",
    "total_price",
    "total_freight_value",
    "total_payment",
    "max_installments",
    "payment_count",
    "distance_km"
]

train[numeric_features].describe().T


,count,mean,std,min,25%,50%,75%,max
total_items,69608.0,1.140171,0.538934,1.00,1.000000,1.000000,1.000000,21.000000
total_price,69608.0,136.140451,205.471452,2.29,45.900000,85.500000,149.900000,13440.000000
total_freight_value,69608.0,22.237072,19.991181,0.00,14.100000,16.790000,23.650000,1002.290000
total_payment,69608.0,159.256581,216.624015,10.07,61.720000,104.350000,175.870000,13664.080000
max_installments,69608.0,2.979485,2.759626,1.00,1.000000,2.000000,4.000000,24.000000
payment_count,69608.0,1.048256,0.412689,1.00,1.000000,1.000000,1.000000,29.000000
distance_km,69608.0,614.151686,593.026318,0.00,226.000339,449.162287,807.712495,5338.619521
